In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def load_gold_table(spark: SparkSession, catalog: str, schema: str, table: str) -> DataFrame:
    """Load a Delta table from the specified catalog.schema.table."""
    return spark.table(f"{catalog}.{schema}.{table}")


def engineer_demand_features(df: DataFrame) -> DataFrame:
    """
    Add lag features, moving average, and calendar features to the demand DataFrame.

    Lag and MA use calendar-day windows (rangeBetween on epoch-day-ordered dates)
    rather than row-based windows, so demand_lag_7 truly means "demand 7 calendar
    days ago" and demand_ma_7 is a true 7-calendar-day moving average — even when
    some dates are missing for a given product-city pair.
    """

    dupe_count = (
        df.groupBy("product_name", "destination_city", "transaction_date")
        .count()
        .filter("count > 1")
        .count()
    )
    if dupe_count > 0:
        raise ValueError(f"Found {dupe_count} duplicate rows for (product_name, destination_city, transaction_date).")

    # Range-based window: orderBy date cast to long (epoch days) so rangeBetween
    # offsets are in calendar days, not row positions.
    w_range = Window.partitionBy("product_name", "destination_city").orderBy(
        F.unix_date("transaction_date")
    )

    # demand_lag_1: demand from exactly 1 calendar day ago (null if that date is missing).
    # demand_lag_7: demand from exactly 7 calendar days ago.
    # max() is safe because the dupe check guarantees at most 1 row per date per
    # partition, so max() returns that single value (or null if no row falls in range).
    # demand_ma_7: average of all rows within the last 7 calendar days (including today).
    return (
        df.withColumns({
            "p_key": F.sha2(F.concat_ws("_", F.col("product_name"), F.col("destination_city")), 256),
            "demand_lag_1": F.max("total_demand_quantity").over(w_range.rangeBetween(-1, -1)),
            "demand_lag_7": F.max("total_demand_quantity").over(w_range.rangeBetween(-7, -7)),
            "demand_ma_7": F.avg("total_demand_quantity").over(w_range.rangeBetween(-6, 0)),
        })
        .select(
            "p_key",
            "transaction_date",
            "product_name",
            "destination_city",
            F.col("total_demand_quantity").alias("total_demand"),
            "avg_unit_price",
            F.col("total_available_inventory").alias("total_inventory"),
            "demand_lag_1",
            "demand_lag_7",
            "demand_ma_7",
            F.dayofweek("transaction_date").alias("day_of_week"),
            F.month("transaction_date").alias("month"),
        )
    )


def write_feature_table(df: DataFrame, catalog: str, schema: str, table: str) -> None:
    """Write the feature DataFrame to a Delta table (overwrite mode)."""
    df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(
        f"{catalog}.{schema}.{table}"
    )

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
gold_table_name = "daily_demand"
feature_table_name = "feature_demand"


In [0]:
df = load_gold_table(spark, catalog_name, gold_schema_name, gold_table_name)
df_new = engineer_demand_features(df)
write_feature_table(df_new, catalog_name, gold_schema_name, feature_table_name)



In [0]:
spark.sql(f"""
    ALTER TABLE {catalog_name}.{gold_schema_name}.{feature_table_name}
    ALTER COLUMN p_key SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {catalog_name}.{gold_schema_name}.{feature_table_name}
    ALTER COLUMN transaction_date SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {catalog_name}.{gold_schema_name}.{feature_table_name}
    ADD CONSTRAINT {feature_table_name}_pk PRIMARY KEY (p_key, transaction_date)
""")

In [0]:
# dupe_count = (
#     df.groupBy("product_name", "destination_city", "transaction_date")
#       .count()
#       .filter("count > 1")
#       .count()
# )

# dupe_count

In [0]:

# display(df_new)

In [0]:
# %sql
# Select * from ct_oil_gas.sc_gold.feature_demand

In [0]:
# df_new.groupBy("pkey", "transaction_date").count().filter("count > 1").show()